# Урок 25. Анализ данных: группировка и агрегация

11 класс · III четверть

[⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/index.ipynb) · [← Урок 24](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/urok-24.ipynb) · [Урок 26 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/urok-26.ipynb)

---

Группировка по признаку, сводные показатели, сравнение групп. Сортировка результатов. Формулировка выводов по числам.

In [ ]:
#@title 🚀 Регистрация { display-mode: "form" }
#@markdown Впиши свои данные и запусти ячейку (Shift+Enter).
#@markdown Если заводил секрет `INFORMATIKA` и включил к нему доступ
#@markdown (🔑 на панели слева) — поля можно оставить пустыми.
#@markdown
#@markdown Colab спросит разрешение на доступ к аккаунту — нажми
#@markdown «Разрешить». Так проверка понимает, чья это работа.
ФИО = "" #@param {type:"string"}
#@markdown Класс — с буквой, например 11А
Класс = "" #@param {type:"string"}

import importlib, urllib.request
urllib.request.urlretrieve("https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/lib/schoolinf.py", "schoolinf.py")
import schoolinf as si
importlib.reload(si)
si.start(lesson="11-25", name=ФИО, klass=Класс)

## Разбираемся

### Главный вопрос анализа данных

Данные сами по себе не отвечают ни на что. Отвечает **сравнение
групп**: где больше, где меньше, где выбивается.

«Средний балл 73» — бесполезное число. «В 9А средний 81, в 9Б —
65, разрыв 16 баллов» — это уже вывод, с которым можно что-то
делать.

Отсюда и схема работы: **сгруппировать → посчитать → сравнить →
сформулировать**.

### Группировка

Группировка — это раскладывание записей по ключу. В коде её делают
словарём:

```python
группы = {}
for запись in записи:
    ключ = запись["класс"]
    if ключ not in группы:
        группы[ключ] = []
    группы[ключ].append(запись)
```

Тот же приём короче — через `setdefault`:

```python
группы.setdefault(ключ, []).append(запись)
```

После этого `группы["9А"]` — список всех записей девятого А,
и дальше с ним работают как с обычным списком.

### Агрегаты

> **Агрегат** — одно число, описывающее всю группу.

| Агрегат | Что показывает | Когда обманывает |
|---|---|---|
| количество | размер группы | — |
| сумма | общий объём | при разных размерах групп |
| среднее | типичное значение | при выбросах |
| **медиана** | серединное значение | почти никогда |
| минимум, максимум | границы | при единичных выбросах |

### Среднее против медианы

Это главная ловушка всей темы. **Среднее** чувствительно
к выбросам, **медиана** — нет.

Зарплаты: 30, 32, 35, 40 и 500 тысяч. Среднее — 127 тысяч,
и такого дохода нет ни у кого. Медиана — 35 тысяч, и это честное
описание.

Правило: если в данных возможны выбросы (доходы, время ответа,
площади), смотрят медиану. Или обе величины сразу: их расхождение
само по себе сигнал.

### Медиана: как считать

1. отсортировать значения;
2. нечётное количество — взять средний элемент;
3. чётное — среднее двух средних.

```python
по_порядку = sorted(значения)
n = len(по_порядку)
if n % 2 == 1:
    медиана = по_порядку[n // 2]
else:
    медиана = (по_порядку[n // 2 - 1] + по_порядку[n // 2]) / 2
```

### Сравнение групп: на что смотреть

* **размеры групп.** Группа из двух человек ничего не доказывает.
* **разброс, а не только среднее.** Два класса со средним 70:
  в одном все по 70, в другом половина по 95 и половина по 45.
* **выбросы.** Один результат в 500 может утащить среднее.

### Формулировка вывода

Вывод — это предложение с числами, которое можно проверить:

* плохо: «9А учится лучше»;
* хорошо: «в 9А медианный балл 81 против 65 в 9Б; при этом в 9Б
  только 12 человек, поэтому разница может быть случайной».

Вторая часть — про осторожность — отличает анализ от гадания.

### Чего данные не говорят

Найденная связь не означает причину. Классический пример:
у детей, спящих со включённым светом, чаще близорукость.
Причина оказалась не в свете: близорукие родители чаще оставляют
свет — и чаще передают близорукость по наследству.

Поэтому честный вывод почти всегда звучит как «связано», а не
«вызывает».

## Смотрим, как это работает

### Пример 1. Данные

In [ ]:
записи = [
    {"фамилия": "Иванов", "класс": "9А", "балл": 92},
    {"фамилия": "Петрова", "класс": "9Б", "балл": 64},
    {"фамилия": "Сидоров", "класс": "9А", "балл": 71},
    {"фамилия": "Кузнецова", "класс": "9Б", "балл": 95},
    {"фамилия": "Орлов", "класс": "9А", "балл": 80},
    {"фамилия": "Никитина", "класс": "9Б", "балл": 61},
    {"фамилия": "Жуков", "класс": "9В", "балл": 53},
    {"фамилия": "Ильина", "класс": "9А", "балл": 88},
]

print("Записей:", len(записи))

### Пример 2. Группировка

In [ ]:
группы = {}
for запись in записи:
    группы.setdefault(запись["класс"], []).append(запись)

for класс in sorted(группы):
    фамилии = [з["фамилия"] for з in группы[класс]]
    print(f"{класс}: {len(фамилии)} чел. — {', '.join(фамилии)}")

### Пример 3. Агрегаты по группам

In [ ]:
def медиана(значения):
    по_порядку = sorted(значения)
    n = len(по_порядку)
    if n == 0:
        return 0
    if n % 2 == 1:
        return по_порядку[n // 2]
    return (по_порядку[n // 2 - 1] + по_порядку[n // 2]) / 2


print(f"{'класс':<7}{'чел.':>5}{'сумма':>7}{'среднее':>9}{'медиана':>9}{'мин':>5}{'макс':>6}")
for класс in sorted(группы):
    баллы = [з["балл"] for з in группы[класс]]
    print(f"{класс:<7}{len(баллы):>5}{sum(баллы):>7}"
          f"{sum(баллы) / len(баллы):>9.1f}{медиана(баллы):>9.1f}"
          f"{min(баллы):>5}{max(баллы):>6}")

Посмотрите на 9В: одна запись, и все агрегаты равны одному числу.
Сравнивать такую группу с остальными нельзя — но формула молчит
об этом, думать приходится самому.

### Пример 4. Среднее врёт

In [ ]:
зарплаты = [30, 32, 35, 40, 500]

print("Значения:", зарплаты)
print(f"Среднее:  {sum(зарплаты) / len(зарплаты):.1f} тыс. — столько не получает никто")
print(f"Медиана:  {медиана(зарплаты):.1f} тыс. — а вот это правда")
print()
print("Расхождение среднего и медианы — сигнал: в данных выброс.")

### Пример 5. Одинаковое среднее, разный разброс

In [ ]:
ровный = [70, 70, 70, 70]
рваный = [95, 95, 45, 45]


def разброс(значения):
    среднее = sum(значения) / len(значения)
    отклонения = [(x - среднее) ** 2 for x in значения]
    return (sum(отклонения) / len(отклонения)) ** 0.5


for название, данные in [("ровный класс", ровный), ("рваный класс", рваный)]:
    print(f"{название}: среднее {sum(данные) / len(данные):.1f}, "
          f"разброс {разброс(данные):.1f}, значения {данные}")

Средние совпали, а картина разная. Разброс (стандартное отклонение)
показывает, насколько значения далеки от среднего.

### Пример 6. Сортировка групп по показателю

In [ ]:
итоги = []
for класс, группа in группы.items():
    баллы = [з["балл"] for з in группа]
    итоги.append((класс, len(баллы), медиана(баллы)))

итоги.sort(key=lambda строка: строка[2], reverse=True)

print("Классы по медианному баллу:")
for класс, сколько, med in итоги:
    пометка = "  ← мало данных" if сколько < 3 else ""
    print(f"  {класс}: {med:>5.1f} ({сколько} чел.){пометка}")

`key=lambda строка: строка[2]` означает «сортируй по третьему
элементу кортежа». `lambda` — короткая безымянная функция;
здесь она нужна только для того, чтобы объяснить сортировке,
что сравнивать.

### Пример 7. Вывод словами

In [ ]:
лучший = итоги[0]
худший = итоги[-1]
надёжные = [и for и in итоги if и[1] >= 3]

print("Вывод:")
print(f"  медианный балл выше всего в {лучший[0]} ({лучший[2]:.1f}),")
print(f"  ниже всего в {худший[0]} ({худший[2]:.1f});")
print(f"  групп с достаточным числом данных: {len(надёжные)} из {len(итоги)}.")
print()
if len(надёжные) < len(итоги):
    print("  Оговорка: часть групп слишком мала, разница может быть случайной.")

## Пробуем сами

Записи приходят списком словарей с полями `класс` и `балл`.

### Задача 1. Устойчивый агрегат

Какой агрегат почти не портится от одного огромного выброса?

In [ ]:
#@title 🧩 Задача 1. Устойчивый показатель { display-mode: "form" }
#@markdown Выбери агрегат
агрегат = "выбери ответ" #@param ["выбери ответ", "среднее", "медиана", "сумма"]

si.ответ("1", агрегат, "9db12c7df2acc529",
         hint="Серединное значение отсортированного ряда.")

### Задача 2. Как называется раскладывание по ключу

Как называется операция раскладывания записей по значению
признака?

In [ ]:
#@title 🧩 Задача 2. Операция { display-mode: "form" }
#@markdown Выбери название
операция = "выбери ответ" #@param ["выбери ответ", "группировка", "сортировка", "фильтрация"]

si.ответ("2", операция, "ab730df88373a962",
         hint="По-английски group by.")

### Задача 3. Группировка

Функция возвращает словарь «класс → список баллов».

In [ ]:
def по_классам(записи):
    return ...

In [ ]:
si.check("3", по_классам, [
    ([{"класс": "9А", "балл": 90}, {"класс": "9Б", "балл": 60},
      {"класс": "9А", "балл": 70}], {"9А": [90, 70], "9Б": [60]}),
    ([{"класс": "9В", "балл": 50}], {"9В": [50]}),
    ([], {}),
])

### Задача 4. Медиана

Функция возвращает медиану списка чисел. Для пустого списка — 0.

Не забудьте про чётное количество элементов.

In [ ]:
def медиана_списка(значения):
    return ...

In [ ]:
si.check("4", медиана_списка, [
    ([30, 32, 35, 40, 500], 35),
    ([1, 2, 3, 4], 2.5),
    ([7], 7),
    ([], 0),
])

### Задача 5. Сводка по группам

Функция возвращает словарь «класс → среднее по классу»,
округлённое до одного знака.

In [ ]:
def средние_по_классам(записи):
    return ...

In [ ]:
si.check("5", средние_по_классам, [
    ([{"класс": "9А", "балл": 90}, {"класс": "9А", "балл": 70},
      {"класс": "9Б", "балл": 60}], {"9А": 80.0, "9Б": 60.0}),
    ([{"класс": "9А", "балл": 1}, {"класс": "9А", "балл": 2}], {"9А": 1.5}),
    ([], {}),
])

### Задача 6. Лучшая группа с оговоркой

Функция возвращает название класса с наибольшим медианным баллом,
но учитывает только классы, где **не меньше трёх** учеников.

Если подходящих классов нет, верните пустую строку.

In [ ]:
def лучший_класс(записи, минимум=3):
    return ...

In [ ]:
si.check("6", лучший_класс, [
    ([{"класс": "9А", "балл": 90}, {"класс": "9А", "балл": 80},
      {"класс": "9А", "балл": 85}, {"класс": "9Б", "балл": 100}], "9А"),
    ([{"класс": "9Б", "балл": 100}], ""),
    ([], ""),
])

Третья проверка — как раз про осторожность: класс с одним
результатом 100 не признаётся лучшим, потому что данных мало.

## Домашнее задание

### Домашнее задание 1. Анализ своего набора данных

Возьмите набор данных из прошлого урока и проведите анализ:

1. сгруппируйте записи по категориальному признаку;
2. посчитайте по каждой группе количество, среднее, медиану,
   минимум и максимум;
3. отсортируйте группы по медиане;
4. напишите **три вывода** предложениями с числами — и к каждому
   оговорку, при каких условиях он может оказаться неверным.

Отдельно найдите группу, где среднее и медиана сильно расходятся,
и объясните, почему. Покажите работу учителю.

### Домашнее задание 2. Размах по группам

Функция возвращает словарь «класс → разница между максимальным
и минимальным баллом».

In [ ]:
def размах_по_классам(записи):
    return ...

In [ ]:
si.check("дз2", размах_по_классам, [
    ([{"класс": "9А", "балл": 90}, {"класс": "9А", "балл": 70}], {"9А": 20}),
    ([{"класс": "9Б", "балл": 50}], {"9Б": 0}),
    ([], {}),
])

### Домашнее задание 3. Медиана чётного ряда

Чему равна медиана ряда 1, 2, 3, 4?

In [ ]:
#@title 🏠 Домашнее 3. Медиана { display-mode: "form" }
#@markdown Впиши число (можно дробное)
med = 0 #@param {type:"number"}

si.ответ("дз3", med, "b8736b9999090496",
         hint="Среднее двух средних элементов.")

---

### Любопытно

В 1854 году Лондон косила холера, и врачи считали, что болезнь
разносится воздухом. Врач Джон Сноу поступил иначе: он нанёс
на карту города каждый случай смерти.

Точки собрались вокруг одной водяной колонки на Брод-стрит.
Сноу добился, чтобы с колонки сняли рукоятку, — и вспышка
прекратилась.

Это считают первым случаем, когда группировка данных
по географическому признаку спасла жизни. Никакой статистики
сложнее подсчёта точек на карте там не было.

---

## Отчёт по уроку

Запусти ячейку ниже, когда решишь задачи. Результат уйдёт учителю автоматически.

In [ ]:
si.report()

---

[← Урок 24](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/urok-24.ipynb) · [⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/index.ipynb) · [🏠 Ко всем классам](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/index.ipynb) · [Урок 26 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/urok-26.ipynb)